# Integrated Gradients for Pairwise Ranking Explanations

This notebook computes Integrated Gradients (IG) attributions for pairwise ranking decisions.

| Method | Description |
|---|---|
| **Pointwise IG** (baseline) | IG computed separately for `s(q,di)` and `s(q,dj)`, then subtracted |
| **Pairwise IG** (proposed) | IG computed w.r.t. `g(q, di, dj) = s(q,di) - s(q,dj)` directly |

Outputs: attribution vectors per token per pair, saved for faithfulness and stability evaluation.

In [76]:
# -- IMPORTS --
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import torch.nn.functional as F
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder
from captum.attr import IntegratedGradients

In [77]:
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
out_dir = Path("../outputs")
pairs_file = out_dir / "pairwise_scores.pkl"

# IG approximates the integral from baseline to input using N interpolated steps.
n_steps = 100

seed = 42
torch.manual_seed(seed)

## Load Model & Tokenizer

Load the cross-encoder and separately load its tokenizer.
Need the tokenizer explicitly because IG operates at the embedding level --> we need to convert token IDs to embedding vectors ourselves so Captum can
compute gradients with respect to them (you can't take gradients w.r.t. discrete integers).

In [78]:
ce_model = CrossEncoder(model_name, max_length=512)
bert_model = ce_model.model
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
bert_model = bert_model.to(device)
bert_model.eval()

print(f"Model device: {device}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

Model device: mps
Tokenizer vocab size: 30522


## Load Pairwise Pairs

In [79]:
pairs_df = pd.read_pickle(pairs_file)
print(f"Pairs loaded: {len(pairs_df)}")
print(f"Queries covered: {pairs_df['qid'].nunique()}")
pairs_df.head(3)

Pairs loaded: 90
Queries covered: 18


,qid,query,pid_i,passage_i,score_i,pid_j,passage_j,score_j,g_score,correct_pref
0,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",10.179455,6875160,Home-schooling in Illinois is considered to be...,-8.177610,18.357065,1
1,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",10.179455,4638437,Tuition and fees at Tulane University of Louis...,-3.351319,13.530774,1
2,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",10.179455,4453731,The average GPA at University of Illinois at C...,-6.636549,16.816004,1


## Tokenize a (query, passage) pair

The cross-encoder expects input formatted as:
`[CLS] query tokens [SEP] passage tokens [SEP]`

We tokenize to get input_ids and attention_mask, then look up the embedding vectors.
IG will attribute over these embedding vectors — one attribution value per token.

We also record which token positions correspond to the query vs the passage,
so we can report query-level and passage-level attributions separately.

In [80]:
def tokenize_pair(query: str, passage: str, max_length: int = 512):
    """
    Tokenizes a (query, passage) pair and returns everything needed for IG.
    Returns a dict with:
      - input_ids, attention_mask (as tensors on device)
      - embeddings: the embedding matrix for this input (shape: 1 x seq_len x hidden)
      - tokens: list of string tokens for display
      - sep_idx: index of the first [SEP] token (end of query)
    """
    encoded = tokenizer(query, passage, max_length=max_length, truncation=True, padding=False, return_tensors="pt")
    
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # look up embedding vectors for each token (detach and require grad so Captum can differentiate through them)
    embedding_layer = bert_model.bert.embeddings
    embeddings = embedding_layer(input_ids).detach().requires_grad_(True)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    # find where the query ends (first [SEP] token)
    sep_positions = [i for i, t in enumerate(tokens) if t == "[SEP]"]
    sep_idx = sep_positions[0] if sep_positions else len(tokens) - 1

    tokenized = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "embeddings": embeddings,
        "tokens": tokens,
        "sep_idx": sep_idx}
    
    return tokenized

## Forward pass through embeddings

Captum's IntegratedGradients needs a function that:
- Takes an embedding tensor as input
- Returns a scalar output (the score we want to attribute)

Can't use the standard `model.predict()` here because that takes raw text. Instead write a forward function that passes pre-computed embeddings directly into the transformer, bypassing the embedding lookup layer.

In [81]:
def forward_from_embeddings(embeddings, attention_mask):
    """
    Runs the BERT model starting from embeddings (not token IDs).
    Returns the scalar relevance logit (shape: (batch,)).

    The normal forward pass: token IDs → embedding lookup → transformer layers → classifier → score
    We skip the embedding lookup and inject embeddings directly, so gradients can flow back to the embedding values.
    """
    outputs = bert_model.bert(inputs_embeds=embeddings,attention_mask=attention_mask)
    
    pooled = outputs.pooler_output
    logit = bert_model.classifier(pooled)
    logit = logit.squeeze(-1)

    return logit

## The Baseline

IG requires a baseline- a reference input representing "no information".
Attributions measure how much each token moves the output **away from the baseline**.

My choice: all-zeros embedding vector.

Why zeros?
- IG is defined at the embedding level, not the token level. A zero vector is a natural "neutral" point in embedding space.
- Using [MASK] token embeddings is also common, but zero is simpler, more principled (it's the actual zero point), and standard in the Captum documentation for transformer models.
- We keep [CLS] and [SEP] at their actual embeddings in both baseline and input; they are structural tokens, not content tokens.

The integral is then approximated as: 
IG(x) ≈ (x - baseline) × mean(gradients at N interpolated points between baseline and x)

In [82]:
def make_baseline(embeddings, input_ids):
    """
    Creates a zero baseline for IG, but preserves the actual embeddings 
    for special tokens [CLS] (pos 0) and [SEP] tokens.
    """
    baseline = torch.zeros_like(embeddings)

    # restore special token embeddings
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    special_ids = {cls_id, sep_id}

    for pos, token_id in enumerate(input_ids[0].tolist()):
        if token_id in special_ids:
            baseline[0, pos, :] = embeddings[0, pos, :].detach()

    return baseline

## Aggregate subword to word level

BERT tokenizes words into subword pieces. IG gives one attribution score per subword token.

We aggregate subword attributions into word-level attributions by summing.
Summing is standard (used in the original IG paper and most follow-up work) because it preserves the total attribution (completeness axiom of IG).

We also return a scalar attribution per token by taking the L2 norm across the hidden dimension (each token has a 384-dim attribution vector).

In [83]:
def aggregate_attributions(attributions, tokens):
    """
    attributions: tensor of shape (1, seq_len, hidden_dim)
    tokens: list of subword token strings

    Returns:
      word_tokens: list of word strings
      word_scores: numpy array of scalar attribution per word
      token_scores: numpy array of scalar attribution per subword token
    """
    # L2 norm across hidden dim -> scalar per subword token
    token_scores = attributions[0].norm(dim=-1).detach().cpu().numpy()

    # merge subword pieces into words
    word_tokens, word_scores = [], []
    current_word, current_score = "", 0.0

    for token, score in zip(tokens, token_scores):
        if token in ("[CLS]", "[SEP]", "[PAD]"):
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
                current_word, current_score = "", 0.0
            # skip special tokens in output
            continue
        elif token.startswith("##"):
            current_word  += token[2:]
            current_score += score
        # start of a new word
        else:
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
            current_word = token
            current_score = score

    if current_word:
        word_tokens.append(current_word)
        word_scores.append(current_score)

    return word_tokens, np.array(word_scores), token_scores

## Pointwise and Pairwise Methods
### Method A: Pointwise IG (baseline)

Compute IG separately for each document w.r.t. its own pointwise score:
- IG_i = IG(s(q, di)) -> attributions for passage i
- IG_j = IG(s(q, dj)) -> attributions for passage j

We then compute a pairwise explanation by subtracting: IG_i - IG_j (aligned by position).

This is the naive baseline from the proposal. 
The key limitation: it treats the two documents independently. The subtraction is done **after** attribution, not during. Therefore, the gradients never "know about" the other document. This may lead to attributions that don't faithfully reflect the pairwise decision.

In [84]:
def compute_pointwise_ig(query, passage):
    """
    Computes standard pointwise IG for a single (query, passage) pair.
    Target function: s(q, d)- the raw relevance score.

    Returns: dict with tokens, word-level attributions, token attributions
    """
    tok = tokenize_pair(query, passage)
    baseline = make_baseline(tok["embeddings"], tok["input_ids"])

    def forward_pointwise(embeds):
        return forward_from_embeddings(embeds, tok["attention_mask"])

    ig = IntegratedGradients(forward_pointwise)

    attributions, delta = ig.attribute(
        inputs=tok["embeddings"],
        baselines=baseline,
        n_steps=n_steps,
        return_convergence_delta=True,
        internal_batch_size=10)

    word_tokens, word_scores, token_scores = aggregate_attributions(attributions, tok["tokens"])

    pointwise_results = {
        "method": "pointwise_ig",
        "tokens": tok["tokens"],
        "word_tokens": word_tokens,
        "word_scores": word_scores,
        "token_scores": token_scores,
        "sep_idx": tok["sep_idx"],
        "convergence_delta": delta.item()}
    
    return pointwise_results

### Method B: Pairwise IG (proposed method)

We define the target function as the log-sigmoid of the preference margin:

  `f(q, di, dj) = log(σ(s(q, di) - s(q, dj)))`

This is the standard pairwise ranking loss (log-sigmoid loss), and we compute IG 
on the input `[query + di]` with respect to this function.

**Why log-sigmoid and not the raw difference?**  
Using `g = s(di) - s(dj)` directly would give identical gradients to pointwise IG 
on di, because the derivative of `f(x) - c` is the same as `f(x)` when c is constant. 
The log-sigmoid is nonlinear, which breaks this equivalence. Its gradient w.r.t. 
s(di) is `σ(1 - σ(g))` — a scaling factor that depends on the margin g itself.

**What this scaling means in practice:**  
- When g is large (easy pair, model very confident): σ(g) ≈ 1, gradient scale ≈ 0.  
  Attribution is spread broadly — no single token is critically decisive.  
- When g is small (tight pair, close decision): σ(g) ≈ 0.5, gradient scale ≈ 0.25.  
  Attribution concentrates on the tokens that tip the decision.

**score_j is computed dynamically** inside the forward pass at each integration step,
not precomputed and held constant. This means the gradient of the target function 
w.r.t. di's tokens genuinely sees dj's score as part of the computation graph.

This makes pairwise IG **margin-aware**: 
- it answers: "which tokens in di explain why di is ranked above dj, given how confident that decision is?"
- rather than pointwise IG's answer of: "which tokens make di look relevant?"

In [85]:
def compute_pairwise_ig(query, passage_i, passage_j):
    """
    Both documents scored inside the forward pass.
    score_j is recomputed at each integration step.
    This means the gradient of g w.r.t. di's tokens sees dj's score as part of the computation graph.
    """
    tok_i = tokenize_pair(query, passage_i)
    tok_j = tokenize_pair(query, passage_j)
    baseline = make_baseline(tok_i["embeddings"], tok_i["input_ids"])

    embeds_j = tok_j["embeddings"].detach()

    def forward_pairwise(embeds):
        score_i = forward_from_embeddings(embeds,   tok_i["attention_mask"])
        score_j = forward_from_embeddings(embeds_j, tok_j["attention_mask"])
        return F.logsigmoid(score_i - score_j)

    ig = IntegratedGradients(forward_pairwise)
    attributions, delta = ig.attribute(
        inputs=tok_i["embeddings"],
        baselines=baseline,
        n_steps=n_steps,
        return_convergence_delta=True,
        internal_batch_size=10)

    word_tokens, word_scores, token_scores = aggregate_attributions(attributions, tok_i["tokens"])

    pairwise_results = {
        "method": "pairwise_ig",
        "tokens": tok_i["tokens"],
        "word_tokens": word_tokens,
        "word_scores": word_scores,
        "token_scores": token_scores,
        "sep_idx": tok_i["sep_idx"],
        "convergence_delta": delta.item()}

    return pairwise_results

## Test on a Single Pair

Before running all pairs, test on one example to verify:
1. The model runs without errors
2. Convergence delta is small (close to 0); this is IG's built-in check that the numerical approximation is accurate enough
3. The top-attributed words make intuitive sense for the query

The convergence delta measures how well the discrete approximation of the integral matches the actual output difference (f(input) - f(baseline)).
A delta close to 0 means the approximation is good. If delta is large, increase n_steps.

In [86]:
# take the first pair with correct preference (g > 0) as the test case
test_row = pairs_df[pairs_df["correct_pref"] == 1].iloc[0]

print(f"Query: {test_row['query']}")
print(f"Passage i: {test_row['passage_i'][:120]}...")
print(f"Passage j: {test_row['passage_j'][:120]}...")
print(f"g score: {test_row['g_score']:.3f} (model prefers di)")

Query: cost of attendance eastern illinois university
Passage i: Eastern Illinois University has roughly 8,000 students. Admission is selective. Tuition is approximately $8,550 per year...
Passage j: Home-schooling in Illinois is considered to be a form of private education. Parents who choose to educate their children...
g score: 18.357 (model prefers di)


In [87]:
# run pointwise IG on di
test_pointwise_i = compute_pointwise_ig(test_row["query"], test_row["passage_i"])
test_pointwise_j = compute_pointwise_ig(test_row["query"], test_row["passage_j"])

print(f"Convergence delta di: {test_pointwise_i['convergence_delta']:.4f}")
print(f"Convergence delta dj: {test_pointwise_j['convergence_delta']:.4f}")

# show top-10 for di (pointwise)
idx = np.argsort(test_pointwise_i["word_scores"])[::-1][:10]
print("\nTop-10 words by pointwise IG attribution (di only):")
for i in idx:
    print(f"{test_pointwise_i['word_tokens'][i]:<20} {test_pointwise_i['word_scores'][i]:.4f}")

Convergence delta di: -0.0000
Convergence delta dj: -0.0002

Top-10 words by pointwise IG attribution (di only):
illinois             2.0460
illinois             1.2017
eastern              1.0287
illinois             0.8517
university           0.7814
tuition              0.5888
attendance           0.5342
eastern              0.5160
tuition              0.5159
university           0.5131


In [88]:
# run pairwise IG
test_pairwise = compute_pairwise_ig(
    test_row["query"],
    test_row["passage_i"],
    test_row["passage_j"])

print(f"Convergence delta: {test_pairwise['convergence_delta']:.4f}")
print(f"Number of word tokens attributed: {len(test_pairwise['word_tokens'])}")

# show top-10 most important words
idx = np.argsort(test_pairwise["word_scores"])[::-1][:10]
print("\nTop-10 words by pairwise IG attribution:")
for i in idx:
    print(f"{test_pairwise['word_tokens'][i]:<20} {test_pairwise['word_scores'][i]:.4f}")

Convergence delta: -0.0000
Number of word tokens attributed: 95

Top-10 words by pairwise IG attribution:
illinois             0.3250
university           0.3036
university           0.2678
has                  0.2359
university           0.2339
cost                 0.2195
tuition              0.2109
illinois             0.1999
tuition              0.1957
students             0.1935


## Run IG on All Pairs

Run both methods on all pairs in `pairwise_scores.pkl`.

For each pair we store:
- The pairwise IG attributions (proposed method)
- The pointwise IG attributions for di and dj separately (baseline method)

Store everything in a list of dicts keyed by (qid, pid_i, pid_j) to easily look up attributions for any pair in the evaluation notebooks.

In [89]:
attribution_records = []
failed_pairs = []

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Computing IG"):
    try:
        # 1) pointwise IG (baseline)
        pt_ig_i = compute_pointwise_ig(row["query"], row["passage_i"])
        pt_ig_j = compute_pointwise_ig(row["query"], row["passage_j"])

        # 2) pairwise IG (proposed)
        pw_ig = compute_pairwise_ig(
            row["query"],
            row["passage_i"],
            row["passage_j"])

        attribution_records.append({
            "qid": row["qid"],
            "query": row["query"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "g_score": row["g_score"],
            "correct_pref": row["correct_pref"],
            # pointwise IG
            "pointwise_ig_i": pt_ig_i,
            "pointwise_ig_j": pt_ig_j,
            # pairwise IG
            "pairwise_ig": pw_ig})

    except Exception as e:
        failed_pairs.append({"qid": row["qid"], "pid_i": row["pid_i"], "error": str(e)})
        print(f"Error on qid={row['qid']}, pid_i={row['pid_i']}: {e}")

print(f"\nSuccessfully attributed: {len(attribution_records)} pairs")
if failed_pairs:
    print(f"Failed: {len(failed_pairs)} pairs")

Computing IG:   0%|          | 0/90 [00:00<?, ?it/s]


Successfully attributed: 90 pairs


## Convergence Check

Check that convergence deltas are small across all pairs.
If the mean delta is large (say > 0.05), consider increasing n_steps.

In [90]:
pw_deltas = [r["pairwise_ig"]["convergence_delta"] for r in attribution_records]
pt_deltas = [r["pointwise_ig_i"]["convergence_delta"] for r in attribution_records]

print("Convergence delta- Pointwise IG (di):")
print(f" mean={np.mean(pt_deltas):.4f}, max={np.max(np.abs(pt_deltas)):.4f}")

print("Convergence delta- Pairwise IG:")
print(f" mean={np.mean(pw_deltas):.4f}, max={np.max(np.abs(pw_deltas)):.4f}")

if np.max(np.abs(pw_deltas)) > 0.05:
    print("\nWarning: some deltas are large. Consider increasing n_steps.")
else:
    print("\nAll deltas look good, approximation is accurate.")

Convergence delta- Pointwise IG (di):
 mean=0.0000, max=0.0011
Convergence delta- Pairwise IG:
 mean=0.0000, max=0.0012

All deltas look good, approximation is accurate.


## Qualitative Preview

Visually inspect examples to confirm attributions look reasonable.
The top-attributed words should relate to the query topic.

In [91]:
def show_top_words(record, method="pairwise_ig", n=10):
    """
    Prints the top-n attributed words for a given pair and method.
    Also splits into query-side and passage-side tokens using sep_idx.
    """
    attr = record[method]
    words = attr["word_tokens"]
    scores = attr["word_scores"]
    idx = np.argsort(scores)[::-1][:n]

    print(f"Method: {method}")
    print(f"Query: {record['query']}")
    print(f"g_score: {record['g_score']:.3f}")
    print(f"\nTop-{n} attributed words:")
    for i in idx:
        print(f" {words[i]:<25} {scores[i]:.4f}")

# show the first record
r = attribution_records[0]
print("=" * 60)
show_top_words(r, method="pointwise_ig_i")
print()
show_top_words(r, method="pairwise_ig")

Method: pointwise_ig_i
Query: cost of attendance eastern illinois university
g_score: 18.357

Top-10 attributed words:
 illinois                  2.0460
 illinois                  1.2017
 eastern                   1.0287
 illinois                  0.8517
 university                0.7814
 tuition                   0.5888
 attendance                0.5342
 eastern                   0.5160
 tuition                   0.5159
 university                0.5131

Method: pairwise_ig
Query: cost of attendance eastern illinois university
g_score: 18.357

Top-10 attributed words:
 illinois                  0.3250
 university                0.3036
 university                0.2678
 has                       0.2359
 university                0.2339
 cost                      0.2195
 tuition                   0.2109
 illinois                  0.1999
 tuition                   0.1957
 students                  0.1935


In [92]:
# find the pair with the smallest g_score (tightest decision)
closest_pair = pairs_df[pairs_df["correct_pref"] == 1].nsmallest(1, "g_score").iloc[0]
print(f"Tightest pair g_score: {closest_pair['g_score']:.3f}")
print(f"Query: {closest_pair['query']}")

# find it in attribution_records
r_tight = next(r for r in attribution_records 
               if r["pid_i"] == closest_pair["pid_i"] 
               and r["pid_j"] == closest_pair["pid_j"])

print("=" * 60)
show_top_words(r_tight, method="pointwise_ig_i")
print()
show_top_words(r_tight, method="pairwise_ig")

Tightest pair g_score: 0.512
Query: what is qualfon
Method: pointwise_ig_i
Query: what is qualfon
g_score: 0.512

Top-10 attributed words:
 qualfon                   3.4949
 outsourcing               1.3950
 qualfon                   1.3899
 outsourcing               0.7538
 contact                   0.7378
 is                        0.6654
 global                    0.5920
 back                      0.4738
 bpo                       0.4656
 a                         0.4365

Method: pairwise_ig
Query: what is qualfon
g_score: 0.512

Top-10 attributed words:
 qualfon                   1.1308
 outsourcing               0.6067
 qualfon                   0.4969
 outsourcing               0.3676
 contact                   0.3095
 is                        0.2623
 global                    0.2535
 provider                  0.2022
 we                        0.2021
 a                         0.1995


In [93]:
# for each pair, compute the ratio of pairwise to pointwise magnitude
# this isolates the margin effect from document-specific effects

for label, r in [("Easy (g=18.357)", attribution_records[0]), ("Tight (g=0.512)", r_tight)]:
    pw_mag = r["pairwise_ig"]["word_scores"].mean()
    pt_mag = r["pointwise_ig_i"]["word_scores"].mean()
    print(f"{label}: pairwise/pointwise magnitude ratio = {pw_mag/pt_mag:.3f}")

Easy (g=18.357): pairwise/pointwise magnitude ratio = 0.363
Tight (g=0.512): pairwise/pointwise magnitude ratio = 0.416


## Save Outputs

In [94]:
out_path = out_dir / "attributions.pkl"
with open(out_path, "wb") as f:
    pickle.dump(attribution_records, f)

print(f"Saved {len(attribution_records)} attribution records → {out_path}")

# save a summary csv
summary = pd.DataFrame([{
    "qid": r["qid"],
    "pid_i": r["pid_i"],
    "pid_j": r["pid_j"],
    "g_score": r["g_score"],
    "correct_pref": r["correct_pref"],
    "pw_ig_delta": r["pairwise_ig"]["convergence_delta"],
    "pt_ig_delta": r["pointwise_ig_i"]["convergence_delta"],
    "pw_top_word": r["pairwise_ig"]["word_tokens"][np.argmax(r["pairwise_ig"]["word_scores"])] if len(r["pairwise_ig"]["word_tokens"]) > 0 else ""} for r in attribution_records])

summary.to_csv(out_dir / "attributions_summary.csv", index=False)
print(f"Saved summary CSV- {out_dir / 'attributions_summary.csv'}")
summary.head()

Saved 90 attribution records → ../outputs/attributions.pkl
Saved summary CSV- ../outputs/attributions_summary.csv


,qid,pid_i,pid_j,g_score,correct_pref,pw_ig_delta,pt_ig_delta,pw_top_word
0,1049774,7185662,6875160,18.357065,1,-0.000030,-0.000014,illinois
1,1049774,7185662,4638437,13.530774,1,-0.000021,-0.000014,illinois
2,1049774,7185662,4453731,16.816004,1,-0.000016,-0.000014,illinois
3,1049774,7185662,4760145,10.647701,1,-0.000018,-0.000014,illinois
4,1049774,7185662,5305640,13.525833,1,-0.000020,-0.000014,illinois
